In [9]:
# ========== 第 1 周 Day 5 变体：从起始页链接 → 筛链 → 抓正文 → 流式研究简报 ==========
# 练习目标：用 Chat Completions + 网页抓取，把某主题（如 AI）整理成结构化 Research Brief
# 和本课关系：system/user messages、response_format=json_object、stream=True + update_display
# 怎么跑：准备 OPENAI_API_KEY；确保同目录 scraper.py 可导入；从上到下运行；最后一格改 topic/URL
# 注意：prompt / model id / URL 保持英文可运行；下方为可执行代码

# 导入标准库 os：读环境变量（例如 OPENAI_API_KEY）
import os
# 导入标准库 json：把模型返回的 JSON 字符串解析成 Python dict
import json
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量
from dotenv import load_dotenv
# 从 IPython.display 导入：display / update_display 用于流式刷新 Markdown
from IPython.display import Markdown, display, update_display
# 从本地 scraper 模块导入：抓页面链接列表、抓页面正文（需同目录有 scraper.py）
from scraper import fetch_website_links, fetch_website_contents
# 从 openai 导入 OpenAI 客户端：调用云端 Chat Completions API
from openai import OpenAI


In [10]:
# ========== 环境 + OpenAI 客户端 ==========

# 加载 .env；override=True 让文件中的值覆盖已有环境变量
load_dotenv(override=True)
# 读取 API Key（本格读出后未强制校验；真正鉴权发生在后续 create 调用）
api_key = os.getenv('OPENAI_API_KEY')

# 默认客户端：密钥从环境变量 OPENAI_API_KEY 自动读取
client = OpenAI()


In [11]:
# ========== System Prompt：链接分析员 —— 只要 JSON，不要闲聊 ==========

# link_analyzer_prompt：发给模型的 system 指令；保留英文以免改变筛选行为与 JSON schema
link_analyzer_prompt = """
You are a skilled research analyst. Your task is to identify the most useful introductory links for a given topic from a list of URLs. 
You must ignore forum posts, product pages, and social media links. Focus on high-quality articles, documentation, and educational resources.
Respond ONLY with a JSON object in the following format:
{
    "links": [
        {"type": "overview_article", "url": "https://..."},
        {"type": "technical_docs", "url": "https://..."},
        {"type": "history_summary", "url": "https://..."}
    ]
}
"""


In [12]:
# ========== System Prompt：情报分析员 —— 把多篇原文压成 Markdown 研究简报 ==========

# briefing_prompt：含 {topic} 占位符，后面用 .format(topic=...) 填入；正文结构要求保留英文
briefing_prompt = """
You are an expert intelligence analyst. You will be given raw text from several articles about a topic. 
Your mission is to synthesize this information into a clear and structured research brief. 
The brief must contain the following sections in Markdown:

Research Brief: {topic}

1. Executive Summary
(A one-paragraph overview of the entire topic.)

2. Key Concepts
(Use bullet points to list and explain the most important terms and ideas.)

3. Important Figures / Events
(List the key people, organizations, or historical events relevant to the topic.)

4. Further Reading
(Provide a list of the original URLs you analyzed for deeper study.)
"""


In [13]:
# ========== 函数：从起始 URL 抓链，让模型筛出与主题最相关的链接（JSON） ==========

def get_relevant_links(topic: str, starting_url: str) -> dict:
    # topic：研究主题；starting_url：入口页（例如 Wikipedia 词条）

    # 获取起始 URL 中的所有链接（依赖 scraper.fetch_website_links）
    links_on_page = fetch_website_links(starting_url)

    # 链接分析的用户提示：把主题、主 URL、链接列表交给模型；prompt 正文保持英文
    user_prompt = f"""
    Please analyze the following links related to the topic "{topic}" and return the most relevant ones for a research brief.
    The main URL is {starting_url}. Make sure all returned URLs are absolute.

    Links:
    {"\n".join(links_on_page)}
    """

    # Chat Completions：system 用分析员人设；response_format 强制 JSON 对象，便于 json.loads
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": link_analyzer_prompt},
            {"role": "user", "content": user_prompt}
        ],
        response_format={"type": "json_object"}
    )

    # 取出助手消息里的 JSON 字符串
    result_json = response.choices[0].message.content
    # 解析为 dict，供后续 get_all_content 遍历 links
    relevant_links = json.loads(result_json)
    return relevant_links


In [14]:
# ========== 函数：按筛出的 URL 逐个抓正文，并附上原始链接列表 ==========

def get_all_content(links_data: dict) -> str:
    # all_content：拼接后的大文本，后面会截断送进简报模型
    all_content = ""
    # original_urls：记住抓过的绝对 URL，供 Further Reading
    original_urls = []

    # 遍历模型返回的 links 列表；缺省用空列表避免 KeyError
    for link in links_data.get("links", []):
        # 每条期望形如 {"type": "...", "url": "https://..."}
        url = link.get("url")
        if url:
            original_urls.append(url)
            # 抓该 URL 的正文（scraper.fetch_website_contents）
            content = fetch_website_contents(url)
            # 用英文标签标记来源，便于模型在简报里引用
            all_content += f"Content from {url} \n{content}\n\n"

    # 文末再附一份原始 URL 清单，方便写 Further Reading
    all_content += f"Original URLs for Reference\n" + "\n".join(original_urls)
    return all_content


In [15]:
# ========== 编排函数：筛链 → 抓正文 → 流式生成研究简报并刷新显示 ==========

def create_research_brief(topic: str, starting_url: str):
    # 第一步：让模型从入口页链接里挑出最有用的子集
    relevant_links = get_relevant_links(topic, starting_url)
    # 第二步：把筛出的页面正文拼成一大段上下文
    full_content = get_all_content(relevant_links)

    # user 消息：要求写 Research Brief；截断到 15000 字符，控制上下文长度与费用
    user_prompt = f"""
    Please create a research brief on the topic "{topic}" using the following content.
    Remember to include the original URLs in the 'Further Reading' section.

    Content:
    {full_content[:15000]}
    """

    # stream=True：边生成边收 delta，而不是等整段结束
    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            # .format(topic=topic)：把 system 模板里的 {topic} 换成真实主题
            {"role": "system", "content": briefing_prompt.format(topic=topic)},
            {"role": "user", "content": user_prompt}
        ],
        stream=True
    )

    # response：累积已生成的 Markdown 文本
    response = ""
    # display_id=True：拿到可更新的显示句柄，后面用 update_display 原地刷新
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        # delta.content 可能为 None（角色/结束块），用 or '' 兜底
        response += chunk.choices[0].delta.content or ''
        # 每来一块就刷新同一 Markdown 区域，形成「打字机」效果
        update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# ========== 入口调用：改 topic / starting_url 即可换主题跑一遍 ==========

# topic 与 Wikipedia URL 保持英文原文（影响抓取目标与简报标题）
create_research_brief(
    topic="The Rise of Artificial Intelligence",
    starting_url="https://en.wikipedia.org/wiki/Artificial_intelligence"
)
